# 01 · Generate synthetic student records

The original analysis used FERPA-protected records from an alternative high school, so the real data can't be shared. This notebook simulates students who resemble that population in aggregate. Notebooks 02 and 03 then run unchanged on these files.

**Two design choices:**

1. **Known effects.** Every student has a hidden *engagement* level that drifts from year to year and drives attendance, course grades, and (more weakly) test scores. Because those relationships are set below, notebook 03 can check whether the model recovers them, which real data can't offer.
2. **Same structure as the raw exports, messiness included.** The output files have the column names, formats, and quirks of the real district exports: duplicated rows, mixed year labels, blank course titles, two overlapping test files, and a few traps for leakage. Each planted issue is listed at the end, so the cleaning in notebook 02 has real work to do.

Calibration targets are rounded aggregates from the real data (no student-level values). No demographic fields are simulated.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 2026
rng = np.random.default_rng(SEED)

OUT_DIR = Path("../synthetic_data")
OUT_DIR.mkdir(exist_ok=True)

N_STUDENTS = 450
COHORTS = np.arange(2013, 2025)  # spring year of each student's (final) senior year
GRADES_START = 2008               # first school year (ending) in the course-history system
ATTENDANCE_START = 2007
TESTS_START = 2004
CREDITS_TO_GRADUATE = 24


def sigmoid(x):
    return 1 / (1 + np.exp(-x))

## 1. The true data-generating process

These parameters are the "ground truth." They are saved to `generating_parameters.json` so notebook 03 can compare model results against them.

In [2]:
TRUE = {
    # engagement: z_t = rho * z_(t-1) + sqrt(1 - rho^2) * noise, one step per school year
    "engagement_rho": 0.85,
    # absence rate = sigmoid(intercept + engagement*z + hs_bump*[grade >= 9] + noise)
    "absence_intercept": -2.75,
    "absence_engagement": -0.65,
    "absence_hs_bump": 0.35,
    "absence_noise_sd": 0.6,
    # P(pass a course) = sigmoid(intercept + engagement*z + term noise + course noise)
    "pass_intercept": 1.75,
    "pass_engagement": 1.35,
    "pass_term_noise_sd": 0.7,
    "pass_course_noise_sd": 0.5,
    # test ability = weight*z + sqrt(1 - weight^2)*(fixed student trait) + test noise
    "test_engagement_weight": 0.5,
    "test_noise_sd": 0.35,
    # paths through school
    "retained_9_intercept": -2.6,       # P(repeat 9th) = sigmoid(intercept + slope*z)
    "retained_9_engagement": -0.9,
    "fifth_year_intercept": -2.4,       # P(fifth year of 12th) = sigmoid(intercept + slope*z)
    "fifth_year_engagement": -0.6,
    "share_enrolled_from_k": 0.55,
    "share_transfer_in_state": 0.6,
    "transfer_credits_missing_rate": 0.15,  # transferred HS credits never entered
}

## 2. Students and their paths through school

Each student gets a cohort, an entry point into the district, and an engagement trajectory. Everyone in the real data reached 12th grade and attended the alternative program, so every simulated student ends there too. Lower engagement moves a student to the alternative program earlier.

In [3]:
def build_student(i):
    cohort = int(rng.choice(COHORTS))
    student_id = int(rng.integers(1_000_000, 2_000_000))

    # engagement trajectory, one value per school year (16 covers K-12 plus repeats)
    rho = TRUE["engagement_rho"]
    z = np.empty(16)
    z[0] = rng.normal()
    for t in range(1, 16):
        z[t] = rho * z[t - 1] + np.sqrt(1 - rho**2) * rng.normal()

    # grade sequence, with an optional repeated 9th grade and fifth senior year
    retained_9 = rng.random() < sigmoid(TRUE["retained_9_intercept"]
                                        + TRUE["retained_9_engagement"] * z[9])
    grades = list(range(0, 10)) + ([9] if retained_9 else []) + [10, 11, 12]
    fifth_year = rng.random() < sigmoid(TRUE["fifth_year_intercept"]
                                        + TRUE["fifth_year_engagement"] * z[len(grades) - 1])
    if fifth_year:
        grades.append(12)

    # how and when the student arrived
    if rng.random() < TRUE["share_enrolled_from_k"]:
        entry_grade, origin = 0, "district"
    else:
        entry_grade = int(rng.integers(1, 12))
        origin = "in_state" if rng.random() < TRUE["share_transfer_in_state"] else "out_of_state"

    alt_entry_grade = int(np.clip(round(11 + 1.0 * z[9] + rng.normal(0, 0.8)), 9, 12))
    alt_entry_grade = max(alt_entry_grade, min(entry_grade, 12))

    return {
        "student_id": student_id,
        "cohort": cohort,
        "grades": grades,
        "z": z[: len(grades)],
        "retained_9": retained_9,
        "fifth_year": fifth_year,
        "entry_grade": entry_grade,
        "origin": origin,
        "alt_entry_grade": alt_entry_grade,
        "elementary": "Riverside Elementary" if rng.random() < 0.7 else "Hillcrest Elementary",
        "transfer_credits_missing": (origin != "district" and entry_grade >= 10
                                     and rng.random() < TRUE["transfer_credits_missing_rate"]),
        "trait": rng.normal(),  # stable test-taking ability, unrelated to engagement
    }


students = [build_student(i) for i in range(N_STUDENTS)]
# student IDs must be unique
assert len({s["student_id"] for s in students}) == N_STUDENTS

Expand each student into one row per school year. `sy` is the school year *ending* (spring year), which is the join key every file will eventually be converted to.

In [4]:
OUT_OF_STATE_SCHOOLS = ["Canyon Ridge High School", "Prairie View High School",
                        "Desert Valley High School", "Pine Hollow High School"]


def school_for(s, grade, in_district):
    if not in_district:
        if s["origin"] == "in_state":
            return "Other In-State Schools"
        return rng.choice(OUT_OF_STATE_SCHOOLS) if grade >= 9 else None
    if grade <= 5:
        return s["elementary"]
    if grade <= 8:
        return "Harbor Middle School"
    return "Northgate Alternative Program" if grade >= s["alt_entry_grade"] else "Northgate High School"


rows = []
for s in students:
    n = len(s["grades"])
    for step, grade in enumerate(s["grades"]):
        sy = s["cohort"] - (n - 1 - step)
        in_district = grade >= s["entry_grade"]
        rows.append({
            "student_id": s["student_id"],
            "sy": sy,
            "step": step,
            "grade": grade,
            "z": s["z"][step],
            "in_district": in_district,
            "first_district_year": in_district and (step == 0 or s["grades"][step - 1] < s["entry_grade"]),
            "school": school_for(s, grade, in_district),
        })
years = pd.DataFrame(rows)
years["alt"] = years["school"].eq("Northgate Alternative Program")
years.head()

,student_id,sy,step,grade,z,in_district,first_district_year,school,alt
0,1178934,2011,0,0,0.240571,True,True,Riverside Elementary,False
1,1178934,2012,1,1,-0.794466,True,False,Riverside Elementary,False
2,1178934,2013,2,2,0.059972,True,False,Riverside Elementary,False
3,1178934,2014,3,3,0.387219,True,False,Riverside Elementary,False
4,1178934,2015,4,4,0.175290,True,False,Riverside Elementary,False


## 3. Attendance

One true row per student per school year, then three artifacts seen in the real export:
- **Phantom second records.** For some years before 2022, an extra row labeled with a single year ("2014", meaning spring 2014) at a *different* school, with near-full membership. Combined, the two rows imply a student enrolled about 340 days in one year, which is impossible.
- **Zero-membership rows** (enrolled on paper, never attended).
- **Heavy exact duplication** (most rows in the real export were copies).

In [5]:
att = years[years["school"].notna() & (years["sy"] >= ATTENDANCE_START)].copy()

full_year = rng.integers(165, 176, len(att))
partial = rng.integers(40, 170, len(att))
att["membership"] = np.where(att["first_district_year"] & (att["grade"] > 0), partial, full_year)
att.loc[rng.random(len(att)) < 0.04, "membership"] = 0

logit = (TRUE["absence_intercept"] + TRUE["absence_engagement"] * att["z"]
         + TRUE["absence_hs_bump"] * (att["grade"] >= 9)
         + rng.normal(0, TRUE["absence_noise_sd"], len(att)))
att["absence_rate_true"] = sigmoid(logit)
att["absent"] = rng.binomial(att["membership"], att["absence_rate_true"]).astype(float)
att["label"] = (att["sy"] - 1).astype(str) + "-" + att["sy"].astype(str)

# a few genuine rows carry the single-year label instead
eligible = att["sy"].between(2008, 2021)
att.loc[eligible & (rng.random(len(att)) < 0.04), "label"] = att["sy"].astype(str)

# phantom second records
other_school = {"Riverside Elementary": "Other In-State Schools",
                "Hillcrest Elementary": "Other In-State Schools",
                "Harbor Middle School": "Riverside Elementary",
                "Northgate High School": "Harbor Middle School",
                "Northgate Alternative Program": "Northgate High School"}
phantom = att[eligible & att["school"].isin(other_school) & (rng.random(len(att)) < 0.30)].copy()
phantom["school"] = phantom["school"].map(other_school)
phantom["membership"] = rng.integers(150, 176, len(phantom))
phantom["absent"] = rng.binomial(phantom["membership"], phantom["absence_rate_true"]).astype(float)
phantom["label"] = phantom["sy"].astype(str)

attendance_clean = pd.concat([att, phantom], ignore_index=True)
attendance_raw = pd.DataFrame({
    "Student ID": attendance_clean["student_id"],
    "Grade": 12,  # the export stamps every row with the grade at extract time
    "Days Absent": attendance_clean["absent"],
    "Days Membership": attendance_clean["membership"],
    "School Year": attendance_clean["label"],
    "OrganizationName": attendance_clean["school"],
})
copies = 1 + rng.poisson(6.8, len(attendance_raw))
attendance_raw = attendance_raw.loc[attendance_raw.index.repeat(copies)]
attendance_raw = attendance_raw.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"true rows: {len(att)}, phantom rows: {len(phantom)}, exported rows: {len(attendance_raw)}")

true rows: 4727, phantom rows: 1149, exported rows: 45730


## 4. Course history

The main high school runs semesters of 0.5-credit courses. The alternative program runs quarters of 0.25-credit courses, with more pass/no-pass grading. Pass probability depends on engagement in that year, plus noise for the term and for the course.

In [6]:
CATALOG = {
    6: ["Language Arts 6", "Math 6", "Science 6", "Social Studies 6", "PE 6", "Focus 6"],
    7: ["Language Arts 7", "Math 7", "Science 7", "Social Studies 7", "PE 7", "Focus 7"],
    8: ["Language Arts 8", "Math 8", "Science 8", "Social Studies 8", "PE 8", "Focus 8"],
    9: ["LA 9", "Algebra 1", "Physical Science", "Global Studies", "Health 1", "PE 9", "Art 1", "Spanish 1"],
    10: ["LA 10", "Geometry", "Biology", "World History", "Health 2", "PE 10", "Art 2", "Spanish 2"],
    11: ["LA 11", "Discrete Math", "Earth Science", "US History", "Chemistry", "Photography", "Woods"],
    12: ["LA 12", "Financial Algebra", "Economics", "Citizenship", "Senior Project", "Photography", "Woods"],
}
ALT_EXTRAS = ["Credit Recovery LA", "Credit Recovery Math", "Discovery", "Enrichment", "Career Prep"]
COURSE_ID = {}


def course_id(title, transferred=False):
    if title not in COURSE_ID:
        COURSE_ID[title] = f"C{len(COURSE_ID) + 1001}"
    return ("TR-" if transferred else "") + COURSE_ID[title]


def terms_for(row):
    fall, spring = row.sy - 1, row.sy
    if row.alt:
        return [("11-November", fall), ("2-February", spring), ("4-April", spring), ("6-June", spring)], 7, 0.25
    if row.grade <= 8:
        return [("1-January", spring), ("6-June", spring)], 6, 0.5
    return [("1-January", spring), ("6-June", spring)], 7, 0.5


def letter(z):
    score = z + rng.normal()
    return "A" if score > 1.1 else "B" if score > 0.3 else "C" if score > -0.7 else "D"


course_rows = []
course_years = years[(years["grade"] >= 6) & (years["sy"] >= GRADES_START)
                     & (years["in_district"] | (years["grade"] >= 9))]
for row in course_years.itertuples():
    transferred = not row.in_district
    terms, n_courses, credit = terms_for(row)
    pool = CATALOG[row.grade] + (ALT_EXTRAS if row.alt else [])
    for month, cal_year in terms:
        term_effect = rng.normal(0, TRUE["pass_term_noise_sd"])
        stopped_attending = row.alt and rng.random() < sigmoid(-3.2 - 1.0 * row.z)
        for title in rng.choice(pool, n_courses, replace=False):
            p_pass = sigmoid(TRUE["pass_intercept"] + TRUE["pass_engagement"] * row.z + term_effect
                             + rng.normal(0, TRUE["pass_course_noise_sd"]))
            passed = rng.random() < p_pass
            pass_fail_graded = row.alt and rng.random() < 0.35
            if stopped_attending:
                mark = "NG"
            elif passed:
                mark = "P" if pass_fail_graded else letter(row.z)
            else:
                mark = "NP" if pass_fail_graded and rng.random() < 0.3 else "F"
                if rng.random() < 0.06:
                    mark = "I"
                elif rng.random() < 0.03:
                    mark = rng.choice(["WF", "W"])
            grade_label = f"{row.grade:02d}"
            if row.grade == 8 and title == "Math 8" and rng.random() < 0.05:
                title, grade_label = "Algebra 1", "08H"  # HS credit earned in 8th grade
            course_rows.append({
                "student_id": row.student_id, "sy": row.sy, "grade": row.grade,
                "transferred": transferred,
                "Calendar Month": month, "cal_year": cal_year,
                "Course ID": course_id(title, transferred), "Grade": grade_label,
                "Course Title": title, "Credit Attempted": credit,
                "Credit Completed": credit if mark in {"A", "B", "C", "D", "P"} else 0.0,
                "Mark": mark,
            })
        if not row.alt and row.grade >= 9 and not transferred and rng.random() < 0.3:
            course_rows.append({
                "student_id": row.student_id, "sy": row.sy, "grade": row.grade, "transferred": False,
                "Calendar Month": "6-June", "cal_year": row.sy, "Course ID": course_id("Advisory"),
                "Grade": f"{row.grade:02d}", "Course Title": "Advisory",
                "Credit Attempted": 0.0, "Credit Completed": 0.0, "Mark": "P",
            })

courses = pd.DataFrame(course_rows)
print(f"course records: {len(courses)}")

course records: 54075


**Ground truth for the outcome.** Diploma credits are counted from the clean records, *including* transferred credits that later go missing from the export. For those students, the label computed from the export will be wrong, which is a real measurement-error risk.

In [7]:
hs = courses[courses["Grade"].isin(["08H", "09", "10", "11", "12"])]
credits_true = hs.groupby("student_id")["Credit Completed"].sum()

truth = pd.DataFrame([{
    "student_id": s["student_id"], "cohort": s["cohort"], "entry_grade": s["entry_grade"],
    "origin": s["origin"], "alt_entry_grade": s["alt_entry_grade"],
    "retained_9": s["retained_9"], "fifth_year": s["fifth_year"],
    "engagement_end_of_9th": s["z"][s["grades"].index(9)],
    "transfer_credits_missing": s["transfer_credits_missing"],
} for s in students]).set_index("student_id")
truth["hs_credits_true"] = credits_true.reindex(truth.index).fillna(0)
truth["earned_24_true"] = (truth["hs_credits_true"] >= CREDITS_TO_GRADUATE).astype(int)
truth["earned_24_true"].mean().round(2)

np.float64(0.44)

Now apply the export's quirks to the course history.

In [8]:
missing_ids = truth.index[truth["transfer_credits_missing"]]
export = courses[~(courses["transferred"] & courses["student_id"].isin(missing_ids))].copy()

# superseded incompletes: same student, term, and course, an earlier "I" row left behind
superseded = export[rng.random(len(export)) < 0.004].copy()
superseded["Mark"], superseded["Credit Completed"] = "I", 0.0
export = pd.concat([export, superseded], ignore_index=True)

# about 11% of course codes lost their titles in the export
code_share = export["Course ID"].value_counts(normalize=True).sample(frac=1, random_state=SEED)
untitled = code_share.index[code_share.cumsum() <= 0.11]
export.loc[export["Course ID"].isin(untitled), "Course Title"] = np.nan

grades_raw = pd.DataFrame({
    "Student ID": export["student_id"],
    "Calendar Month": export["Calendar Month"],
    "Calendar Year": export["cal_year"].astype(str) + "-" + export["cal_year"].astype(str),
    "Course ID": export["Course ID"],
    "Grade": export["Grade"],
    "Course Title": export["Course Title"],
    "Credit Attempted": export["Credit Attempted"],
    "Credit Completed": export["Credit Completed"],
    # the course system labels a school year by its FALL year: "2015-2015" = 2015-16
    "School Year": (export["sy"] - 1).astype(str) + "-" + (export["sy"] - 1).astype(str),
    "Mark": export["Mark"],
})
grades_raw.loc[rng.choice(len(grades_raw), 20, replace=False), "Grade"] = np.nan
grades_raw.loc[rng.choice(len(grades_raw), 6, replace=False), "Calendar Year"] = np.nan

dup_idx = rng.choice(len(grades_raw), int(0.36 * len(grades_raw)), replace=True)
grades_raw = pd.concat([grades_raw, grades_raw.iloc[dup_idx]], ignore_index=True)
grades_raw = grades_raw.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"exported course rows: {len(grades_raw)}")

exported course rows: 73688


## 5. State tests

Tests are given in grades 3-8 and 11 to students enrolled in-state. A legacy state test runs through spring 2014, Smarter Balanced (SBA) from 2015. The two use different scales, so scale scores can't be compared across them; performance levels are the safer common measure. Test results depend on engagement only partly (weight 0.5). The rest is a stable trait unrelated to engagement.

The export also includes graduation-requirement *completion* records (resume, education plan, career experiences), which are more common among students who earned their credits. They aren't tests, and using them as features would be leakage.

In [9]:
SUBSCORES = {
    "LEG Reading": ["Vocabulary", "Demonstrate General Understanding", "Develop an Interpretation"],
    "LEG Math": ["Calculations & Estimation", "Measurement", "Geometry", "Algebraic Relationships"],
    "LEG Science": ["Life Science", "Physical Science", "Earth Science"],
    "SBA ELA": ["Reading", "Writing", "Speaking/Listening", "Research"],
    "SBA MA": ["Concepts and Procedures", "Problem Solving/ Modeling and Data Analysis",
               "Communicating Reasoning"],
}
BLANK = "\t"  # the real export stores blanks as tab characters
trait = {s["student_id"]: s["trait"] for s in students}
w = TRUE["test_engagement_weight"]


def test_name(family, grade, sy):
    if family == "Science":
        return f"LEG Science, Grade {grade:02d}" if sy <= 2018 else "State Science HS"
    if sy <= 2014:
        subject = "LEG Reading" if family == "ELA" else "LEG Math"
        return f"{subject}, Grade {'09-12' if grade == 11 else f'{grade:02d}'}"
    subject = "SBA ELA" if family == "ELA" else "SBA MA"
    return f"{subject} {'HS' if grade == 11 else f'Gr {grade}'}"


test_rows = []
tested = years[years["grade"].isin([3, 4, 5, 6, 7, 8, 11]) & (years["sy"] >= TESTS_START)
               & (years["in_district"] | years["school"].eq("Other In-State Schools"))]
for row in tested.itertuples():
    families = ["ELA", "Math"] + (["Science"] if row.grade in (5, 8, 11) else [])
    for family in families:
        if rng.random() > sigmoid(2.0 + 0.6 * row.z):
            continue  # did not test (absent, opted out)
        ability = w * row.z + np.sqrt(1 - w**2) * trait[row.student_id] + rng.normal(0, TRUE["test_noise_sd"])
        name = test_name(family, row.grade, row.sy)
        admin = pd.Timestamp(year=row.sy, month=int(rng.choice([4, 5])), day=int(rng.integers(1, 29)))
        sba = name.startswith("SBA")
        if sba:
            total = round(2300 + 40 * row.grade + 80 * ability)
            level = str(int(np.digitize(ability, [-0.6, 0.1, 0.9]) + 1))  # 1-4
            levels = {"Performance Level for Benchmark": level, "Performance Level for Grade": BLANK}
        else:
            total = round(200 + 3 * row.grade + 8 * ability)
            level = "D" if ability < -0.2 else "M" if ability < 1.0 else "E"  # Does not meet / Meets / Exceeds
            levels = {"Performance Level for Benchmark": level, "Performance Level for Grade": level}
        test_rows.append({"student_id": row.student_id, "Admin Date": admin, "Test Name": name,
                          "Part Number": 1, "Part Description": "Total", "Scale Score": str(total), **levels})
        prefix = name.split(",")[0] if not sba else ("SBA ELA" if "ELA" in name else "SBA MA")
        for k, part in enumerate(SUBSCORES.get(prefix, []), start=2):
            sub = ability + rng.normal(0, 0.4)
            if sba:
                score = BLANK  # SBA reports claim levels, not scale scores
                sub_levels = {"Performance Level for Benchmark": str(int(np.digitize(sub, [-0.4, 0.6]) + 1)),
                              "Performance Level for Grade": BLANK}
            else:
                score = str(round(200 + 3 * row.grade + 8 * sub))
                sub_level = "D" if sub < -0.2 else "M" if sub < 1.0 else "E"
                sub_levels = {"Performance Level for Benchmark": sub_level, "Performance Level for Grade": sub_level}
            test_rows.append({"student_id": row.student_id, "Admin Date": admin, "Test Name": name,
                              "Part Number": k, "Part Description": part, "Scale Score": score, **sub_levels})

# graduation-requirement completion records (leakage trap), cohorts 2020+
observed_credits = (grades_raw.drop_duplicates()
                    .loc[lambda d: d["Grade"].isin(["08H", "09", "10", "11", "12"])]
                    .groupby("Student ID")["Credit Completed"].sum())
for sid, cohort in truth.loc[truth["cohort"] >= 2020, "cohort"].items():
    on_track = observed_credits.get(sid, 0) >= CREDITS_TO_GRADUATE
    for name in ["Resume Completion", "Education Plan Survey",
                 "PLR - Career Related Exp 1", "PLR - Career Related Exp 2"]:
        if rng.random() < (0.85 if on_track else 0.35):
            test_rows.append({"student_id": sid, "Test Name": name, "Part Number": 1,
                              "Admin Date": pd.Timestamp(year=cohort, month=int(rng.integers(1, 6)), day=15),
                              "Part Description": "Total", "Scale Score": BLANK,
                              "Performance Level for Benchmark": "C", "Performance Level for Grade": "C"})

tests = pd.DataFrame(test_rows)
print(f"test records: {len(tests)}")

test records: 26630


**Enter and Leave dates.** These columns' meaning isn't documented in the real export. Here they're simulated as dates entering and leaving the alternative program. Leave dates are more often filled for students who finished, which makes them another leakage trap.

**Two test files.** The real tests came in two overlapping exports. File A holds mostly older cohorts and stops after 2016. File B covers all years. A few students appear in both, and both files contain duplicate rows.

In [10]:
by_id = {s["student_id"]: s for s in students}
dates = {}
for sid, s in by_id.items():
    alt_fall = s["cohort"] - (len(s["grades"]) - 1 - s["grades"].index(s["alt_entry_grade"])) - 1
    enter = pd.Timestamp(year=alt_fall, month=9, day=1) + pd.Timedelta(days=int(rng.integers(0, 120)))
    finished = observed_credits.get(sid, 0) >= CREDITS_TO_GRADUATE
    if finished and rng.random() < 0.7:
        leave = pd.Timestamp(year=s["cohort"], month=6, day=12)
    elif not finished and rng.random() < 0.45:
        leave = pd.Timestamp(year=s["cohort"] - 1, month=10, day=1) + pd.Timedelta(days=int(rng.integers(0, 230)))
    else:
        leave = pd.NaT
    dates[sid] = (enter, leave)

tests["Enter Date"] = tests["student_id"].map(lambda sid: dates[sid][0])
tests["Leave Date"] = tests["student_id"].map(lambda sid: dates[sid][1])
tests["Grade"] = 12
tests = tests.rename(columns={"student_id": "Student ID"})
TEST_COLUMNS = ["Student ID", "Grade", "Enter Date", "Leave Date", "Admin Date", "Test Name",
                "Part Number", "Part Description", "Scale Score",
                "Performance Level for Benchmark", "Performance Level for Grade"]
tests = tests[TEST_COLUMNS]

draw = rng.random(N_STUDENTS)
file_of = {}
for (sid, s), u in zip(by_id.items(), draw):
    file_of[sid] = "both" if u < 0.07 else "A" if (s["cohort"] <= 2018 and u < 0.60) else "B"
which = tests["Student ID"].map(file_of)
tests_a = tests[which.isin(["A", "both"]) & (tests["Admin Date"].dt.year <= 2016)]
tests_b = tests[which.isin(["B", "both"])]


def add_copies(df, share):
    idx = rng.choice(len(df), int(share * len(df)), replace=True)
    out = pd.concat([df, df.iloc[idx]], ignore_index=True)
    return out.sample(frac=1, random_state=SEED).reset_index(drop=True)


tests_a, tests_b = add_copies(tests_a, 0.45), add_copies(tests_b, 0.30)
print(f"file A rows: {len(tests_a)}, file B rows: {len(tests_b)}")

file A rows: 12372, file B rows: 25158


## 6. Save

In [11]:
grades_raw.to_excel(OUT_DIR / "grades.xlsx", index=False)
attendance_raw.to_csv(OUT_DIR / "attendance.csv", index=False)
tests_a.to_excel(OUT_DIR / "tests_a.xlsx", index=False)
tests_b.to_excel(OUT_DIR / "tests_b.xlsx", index=False)

# ground truth: for checking results in notebook 03, never as model features
truth.reset_index().to_csv(OUT_DIR / "_truth_students.csv", index=False)
with open(OUT_DIR / "generating_parameters.json", "w") as f:
    json.dump(TRUE, f, indent=2)

sorted(p.name for p in OUT_DIR.iterdir())

['_truth_students.csv',
 'attendance.csv',
 'generating_parameters.json',
 'grades.xlsx',
 'tests_a.xlsx',
 'tests_b.xlsx']

## 7. Sanity check against the real data

The real column reports rounded aggregates from profiling the real exports. The simulated column is computed from the files just written, after removing exact duplicates, which is the first cleaning step in notebook 02.

In [12]:
g = pd.read_excel(OUT_DIR / "grades.xlsx")
a = pd.read_csv(OUT_DIR / "attendance.csv")
g1, a1 = g.drop_duplicates(), a.drop_duplicates()
hs1 = g1[g1["Grade"].isin(["08H", "09", "10", "11", "12"])].groupby("Student ID")["Credit Completed"].sum()
rate = (a1["Days Absent"] / a1["Days Membership"]).where(a1["Days Membership"] > 0)
marks = g1["Mark"].value_counts(normalize=True)
single = ~a1["School Year"].astype(str).str.contains("-")

check = pd.DataFrame([
    ("students", "about 450", g["Student ID"].nunique()),
    ("cohorts", "2013-2024, ~30-45 each",
     f"{truth['cohort'].min()}-{truth['cohort'].max()}, {truth['cohort'].value_counts().min()}-{truth['cohort'].value_counts().max()} each"),
    ("share of course rows that are exact copies", "about 25%", f"{1 - len(g1) / len(g):.0%}"),
    ("share of attendance rows that are exact copies", "about 85%", f"{1 - len(a1) / len(a):.0%}"),
    ("median HS credits (after dedup)", "about 23", round(hs1.median(), 1)),
    ("share with 24+ HS credits", "about 40-50%", f"{(hs1 >= 24).mean():.0%}"),
    ("most common mark", "F, about 20-30%", f"{marks.idxmax()}, {marks.max():.0%}"),
    ("share of marks that are P", "about 15%", f"{marks.get('P', 0):.0%}"),
    ("course rows with no title", "about 11%", f"{g1['Course Title'].isna().mean():.0%}"),
    ("median absence rate", "about 0.07", round(rate.median(), 2)),
    ("90th percentile absence rate", "about 0.2", round(rate.quantile(0.9), 2)),
    ("attendance rows with single-year labels", "about 25%", f"{single.mean():.0%}"),
], columns=["measure", "real data (rounded)", "simulated"])
check

,measure,real data (rounded),simulated
0,students,about 450,450
1,cohorts,"2013-2024, ~30-45 each","2013-2024, 30-47 each"
2,share of course rows that are exact copies,about 25%,27%
3,share of attendance rows that are exact copies,about 85%,87%
4,median HS credits (after dedup),about 23,23.2
5,share with 24+ HS credits,about 40-50%,43%
6,most common mark,"F, about 20-30%","F, 19%"
7,share of marks that are P,about 15%,14%
8,course rows with no title,about 11%,10%
9,median absence rate,about 0.07,0.07


## Planted issues for notebook 02

| Issue | Where | What a correct cleaning step does |
|---|---|---|
| Exact duplicate rows | all four files | drop them first |
| `Grade` is always 12 | attendance, tests | ignore it; get grade levels from course history |
| School-year labels differ | grades: fall year ("2015-2015" = 2015-16); attendance: "2014-2015" or "2015" (spring year) | convert everything to school year ending |
| Phantom second attendance records | single-year rows that share a year with a range-format row | keep the range-format row |
| Zero-membership rows | attendance | exclude from absence rates |
| Superseded incompletes | same student, term, course; one row marked "I" | keep the final mark |
| Blank course titles | ~11% of course codes | keep the rows; they still carry credit |
| "P" marks | ~15% of marks | count as passing, not as 0 |
| Missing transferred credits | some transfer-in students | flag; the label is likely wrong for them |
| "08H" grade label | a few 8th-grade Algebra 1 records | decide whether they count as HS credit |
| Blank scores stored as tabs | tests | convert to missing |
| Two test scales | legacy (to 2014) vs SBA (2015+) | use performance levels, or standardize within test |
| Completion records mixed into tests | Resume, Education Plan, PLR rows | exclude: graduation requirements, not tests |
| Enter/Leave dates | tests | exclude: recorded at or after the outcome |
| Overlapping test files | tests_a, tests_b | combine, then drop duplicates |